# 04 Interpretation

## 1. Summary Table of Main Inferential Results

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_combined_processed.csv"
TAB_DIR = ROOT / "outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

BEHAVIOR_VAR = "SadOrHopeless"
BEHAVIOR_P0 = 0.30
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0

processed = pd.read_csv(PROCESSED_PATH)

# Formal behavior inference
sad_binary = processed["sad_binary"].dropna().astype(int)
n_b = int(sad_binary.shape[0])
x_b = int((sad_binary == 1).sum())
phat = x_b / n_b
ci_b = stats.binomtest(x_b, n_b).proportion_ci(confidence_level=0.95, method="wilson")
z_stat = (phat - BEHAVIOR_P0) / np.sqrt(BEHAVIOR_P0 * (1 - BEHAVIOR_P0) / n_b)
p_value_b = 2 * (1 - stats.norm.cdf(abs(z_stat)))
decision_b = "reject H0" if p_value_b < 0.05 else "fail to reject H0"

behavior_summary = pd.DataFrame({
    "analysis": ["Population proportion"],
    "variable": [BEHAVIOR_VAR],
    "estimate": [f"{phat:.4f}"],
    "benchmark": [f"{BEHAVIOR_P0:.4f}"],
    "test_statistic": [f"{z_stat:.4f}"],
    "p_value": ["< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}"],
    "ci_low": [f"{ci_b.low:.4f}"],
    "ci_high": [f"{ci_b.high:.4f}"],
    "decision_5pct": [decision_b]
})

# Formal continuous inference
w = processed[CONT_VAR].dropna()
n_w = int(w.shape[0])
mean_w = float(w.mean())
sd_w = float(w.std(ddof=1))
se_w = sd_w / np.sqrt(n_w)
t_crit = stats.t.ppf(0.975, df=n_w - 1)
ci_low_w = mean_w - t_crit * se_w
ci_high_w = mean_w + t_crit * se_w
t_stat, p_value_w = stats.ttest_1samp(w, popmean=CONT_MU0)
decision_w = "reject H0" if p_value_w < 0.05 else "fail to reject H0"

mean_summary = pd.DataFrame({
    "analysis": ["Population mean"],
    "variable": [CONT_VAR],
    "estimate": [f"{mean_w:.4f}"],
    "benchmark": [f"{CONT_MU0:.4f}"],
    "test_statistic": [f"{t_stat:.4f}"],
    "p_value": ["< 0.0001" if p_value_w < 0.0001 else f"{p_value_w:.4f}"],
    "ci_low": [f"{ci_low_w:.4f}"],
    "ci_high": [f"{ci_high_w:.4f}"],
    "decision_5pct": [decision_w]
})

inference_summary = pd.concat([behavior_summary, mean_summary], ignore_index=True)
inference_summary.to_csv(TAB_DIR / "03_inference_summary_table.csv", index=False)
display(inference_summary)

,analysis,variable,estimate,benchmark,test_statistic,p_value,ci_low,ci_high,decision_5pct
0,Population proportion,SadOrHopeless,0.3000,0.3000,-0.0093,0.9926,0.2924,0.3077,fail to reject H0
1,Population mean,HowMuchDoYouWeighWithoutShoesInKG,68.5502,68.0000,3.7007,0.0002,68.2588,68.8416,reject H0


**Interpretation:** The two formal one-sample analyses lead to different conclusions. For **SadOrHopeless**, the estimated proportion is essentially equal to the benchmark of **0.30**, the **95% confidence interval** contains **0.30**, and the null hypothesis is **not rejected** at the 5% level. For **HowMuchDoYouWeighWithoutShoesInKG**, the estimated mean is slightly above **68.0 kg**, the **95% confidence interval** stays above **68.0**, and the null hypothesis is **rejected** at the 5% level.

## 2. Final Synthesis

For the **behavior variable**, the EDA showed that the **No** category was much more common than the **Yes** category, and the recoded success proportion looked visually close to **0.30**. The formal proportion analysis confirmed this pattern: the sample proportion was **0.3000**, the **95% confidence interval** was approximately **(0.2924, 0.3077)**, and the one-sample test produced a very large **p-value**. Therefore, the sample does not provide enough evidence that the population proportion of **SadOrHopeless** is different from **0.30**. A cautious point is that this result does not prove the true population proportion is exactly **0.30**; it only means the sample does not provide enough evidence that it is different from **0.30**.

For the **continuous variable**, the EDA showed a large sample size, moderate spread, and some right-skewness in the weight distribution. The formal mean analysis found a sample mean of **68.5502 kg**, with a **95% confidence interval** of approximately **(68.2588, 68.8416)**. Because the interval stays above **68.0 kg** and the **p-value** is about **0.0002**, the null hypothesis is rejected at the 5% significance level. In context, the average body weight in this sample appears to be slightly above the benchmark, although the practical difference is not very large.

As an **exploratory extension**, we also examined whether **SadOrHopeless** was more common among students who reported smoking on at least 1 day in the past 30 days, and whether this pattern varied across age groups. The additional EDA suggested that current smokers tended to show a higher **SadOrHopeless** proportion than non-smokers across most age groups, and that the smoker-minus-non-smoker gap stayed positive in every age group. Among current smokers, the smoking-frequency distribution also appeared to differ somewhat by **SadOrHopeless** status. These extension results should be interpreted as **descriptive and exploratory**, rather than as formal confirmatory or causal findings.

Overall, the formal analyses show that the **SadOrHopeless** proportion is not detectably different from its benchmark of **0.30**, while the average body weight is slightly but significantly above **68.0 kg**. At the same time, the additional exploratory analysis suggests a broader positive association between smoking-related behavior and **SadOrHopeless** across age groups.